# `ObsFile` observation file class sandbox

Sandbox to develop and test the `ObsFile` observation file handling class for DEMONEXT

Uses YAML (YAML Ain't Markup Language), a simple, structured, ascii-based data serialization language used
across many languages and operating systems to create human-readable and editable configuration files.

We use YAML in DEMONEXT for the obsevatory control and data-taking system runtime configuration file
and for observation specification files.

## `ObsFile` class

The `ObsFile` class implements a YAML-based observation file.  This notebook is our development and
testing sandbox before we integrate the new class into the `demonext` module.

## ObsFile format

Observation files are YAML formatted, here is a maximalist example:

### Multi-filter sequence

<pre>
#
# DEMONEXT 2025 observation file
# Generated: 2025 Jan 29, mkObs.py v0.2.3
#
project:
  PROJECT: OSU_TESS
  PI_NAME: Carden
  PI_INST: OSU
  PI_EMAIL: carden.33@osu.edu
  
target:
  Name: M101
  RA2000: '14:03:12.3'
  Dec2000: '+54:20:57'
  GuideMode: auto
  Priority: 1
  
sequence:
  NumObs: 1
  obs1: ["g_sdss",60.0,10]
  obs2: ["r_sdss",60.0,10]
  obs3: ["i_sdss",60.0,10]

constraints: {}

status:
   parsed: Y
   validated: Y
</pre>

### Multi-exposure, single filter sequence

<pre>
project:
  PROJECT: OSU_TESS
  PI_NAME: Carden
  PI_INST: OSU
  PI_EMAIL: carden.33@osu.edu

target:
  Name: TOI-3912 b
  RA2000: '14:23:12.4'
  Dec2000: '+24:04:35.6'
  GuideMode: science
  Priority: 1

sequence:
  NumObs: 1
  obs1: ["r_sdss",120.0,140]

constraints:
  StartTime: '2025-04-29T05:30:00Z'
  EndTime: '2025-04-29T11:08:00Z'
  
status:
  parsed: Y
  validated: N
</pre>


### Dependencies

The `obsfile.py` code requires the following:
 * YAML
 * logging
 * astropy: coordinates.SkyCoord, coordinates.TETE, units, and time.Time
 * os

### Info about YAML

 * Official YAML website: https://yaml.org/
 * PyYAML, the standard python YAML loader/emitter (comes with Anaconda): http://pyyaml.org 
 * A nice YAML tutorial: https://www.cloudbees.com/blog/yaml-tutorial-everything-you-need-get-started
 * Another YAML tutorial: https://python.land/data-processing/python-yaml


In [1]:
import obsfile

### Open and dump contents

Opens the named obs file and prints the contents to the screen by hand.

In [20]:
obsFile = "OSU_TESS_TOI-3912b.obs"
obsFile = "OSU_TESS_M101.obs"

try:
    obs = obsfile.ObsFile(obsFile)
    obs.print()
except Exception as exp:
    print(f"ERROR: loadConfig() - {exp}")



Observation file OSU_TESS_M101.obs contents:

project:
  PROJECT: OSU_TESS
  PI_NAME: Carden
  PI_INST: OSU
  PI_EMAIL: carden.33@osu.edu

target:
  Name: M101 dw1
  RA2000: 14:03:12.3
  Dec2000: +54:20:57
  Priority: 1
  GuideMode: auto

sequence:
  NumObs: 1
  obs1: ['g_sdss', 60.0, 10]
  obs2: ['r_sdss', 60.0, 10]
  obs3: ['i_sdss', 60.0, 10]

constraints:
  Airmass: 1.5

status:
  parsed: Y
  validated: N


## Open and parse info

Opens `myObs.obs` and shows how to get at the contents through the class properties to 
create a summary of the observation.

### Object Coordinates

`ObsFile` expects J2000 equinox coordinates and computes the apparent RA/Dec in decimal hours/degrees,
respectively at the current instant (when the class is instantiated).  This uses a minimalist
version of the `precess()` method in the `Telescope` class, but it knows to expect sexigesimal RA/Dec.
It is reasonably agnostic provided the coordinates are RA in hours and Dec in degrees.

### Observing parameters (obsPars)

The parameters to acquire 1 or more science images though a given filter of a given exposure time
is an `obsPars` list, defined in the YAML observation file like
 > `obs1: ['g_SDSS',90,3]`

in order: `filterID`, `expTime`, and `numImages`.  The code snippet below shows how to read/parse one of these.

### code snippet

Using this with the camera object might look like this in autoguiding mode (not executable in this notebook)
```python
    import demonext
    from demonext import camera, obsfile, telescope,focuser    
    
    tel = telescope.Telescope(cfgFile)
    foc = focuser.Focuser(cfgFile)
    cam = camera.Camera(cfgFile)
    
    # ... general setup stuff
    
    # Load the observation file
    
    obsFile = "myObs.obs"
    obs = obsfile.ObsFile(obsFile)
    
    # pass project info and target name to the camera object
    
    cam.projInfo = obs.projInfo
    cam.objectID(obs.name)
    
    # point the telescope to the target ...
    
    try:
        tel.slewToRADec(obs.appRA,obs.appDec)
    except Exception as exp:
        print(f"oops: {exp}")
        # abort observation
        
    # focus the telescope ...
    
    # start the autoguider ...
    
    if cam.guidingOn():
        print(f"autoguider started...")
    else:
        print(f"ERROR: autoguider start failed: {cam.msg}")
        # abort observation
        
    # execute the observing sequence
    
    for seqNum in range(obs.repeat+1): # number of repetitions, could be 0 which means 1 sequence w/o reps
        print(f"Starting observing sequence {seqNum+1} of {obs.repeat+1}:")
        for pars in obs.obsPars:
            try:
                cam.observe(pars)  # pass pars to the Camera.observe() method
            except Exception as exp:
                print(f"oops: {exp}")
                # abort observation
    
    print("Done: observing sequence completed")
```

In [18]:
# try something

obsFile = 'OSU_TESS_M101.obs'
#obsFile = 'OSU_TESS_TOI-3912b.obs'

try:
    obs = obsfile.ObsFile(obsFile)

    print(f"Target: {obs.name}")
    print(f"     J2000: {obs.RA} {obs.Dec}")
    print(f"  Apparent: {obs.appRA}h {obs.appDec}d")
    print(f"Guide Mode: {obs.guideMode}")
    print(f"  Priority: {obs.priority}")
    
    print(f"\nImage Sequence:")
    for i, pars in enumerate(obs.obsPars):
        filt = pars[0]
        expT = pars[1]
        numExp = pars[2]
        print(f"  {filt} filter, {numExp}x{expT:.1f} sec")
    if obs.numObs > 0:
        print(f"  Execute sequence {obs.numObs} times")
    else:
        print("  Execute sequence once")

    if len(obs.conInfo) > 0:
        print("\nConstraints:")
        if obs.airmass:
            print(f"  Airmass < {obs.airmass:.1f}")
        if obs.moonPhase:
            print(f"  MoonPhase < {100*obs.moonPhase:.0f}% illuminated")
        if obs.moonAngle:
            print(f"  Moon Angle < {obs.moonAngle:.1f} degrees")
        if obs.maxSky:
            print(f"  Sky Counts < {obs.maxSky:.1f} adu")
        if obs.startTime:
            print(f"  Start after {obs.startTime}")
        if obs.endTime:
            print(f"  End before {obs.endTime}")
            
    else:
        print("\nConstraints: None")
        
except Exception as exp:
    print(f"ERROR: {exp}")

Target: M101
     J2000: 14:03:12.3 +54:20:57
  Apparent: 14.069137644331814h 54.226575044239254d
Guide Mode: auto
  Priority: 1

Image Sequence:
  g_sdss filter, 10x60.0 sec
  r_sdss filter, 10x60.0 sec
  i_sdss filter, 10x60.0 sec
  Execute sequence 1 times

Constraints:
  Airmass < 1.5
